In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ModuleNotFoundError: No module named 'torch'

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=10, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(10 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = x.view(-1, 10 * 14 * 14)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

In [ ]:
test_loader = DataLoader(datasets.MNIST('./data', train=False, download=True, transform=transform), batch_size=64, shuffle=False)


In [ ]:
model = MNISTNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train(model, train_loader, optimizer):
    model.train()
    t_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        t_loss = t_loss + loss.item()

    return t_loss/len(train_loader)

In [ ]:
def test(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            if batch_idx % 20 == 0:
              plt.imshow(data[0].view(28, 28).numpy(), cmap='gray')
              plt.title(f"Predicted: {pred[0].item()}, Actual: {target[0].item()}")
              plt.show()

    test_loss /= len(test_loader.dataset)

    print(f'Test accuracy: {100. * correct / len(test_loader.dataset):.4f}%')
    print(f'Test loss: {test_loss:.4f}')

In [ ]:
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    loss = train(model, train_loader, optimizer)

    print(f'Epoch: {epoch}, Loss: {loss:.4f}')

In [ ]:
test(model, test_loader)